# Duckietown SDK - Hello World Tutorial

Welcome to the Duckietown SDK! This notebook will guide you through the basics of controlling a Duckiebot using Python.

## What you'll learn:
- How to connect to a Duckiebot (real or simulated)
- How to control the motors to move the robot
- How to control the LEDs to make pretty light patterns
- How to read sensor data from cameras and wheel encoders
- How to get the robot's pose (position and orientation)

Let's get started! 🚗🦆

## 1. Setting up the Environment

First, let's import the necessary modules:

In [ ]:
# Import the main robot class
from duckietown.sdk.robots.duckiebot import DB21J

# Import message types for controlling lights
from duckietown_messages.actuators import CarLights
from duckietown_messages.colors import RGBA

# Import utilities
import time
from typing import Tuple

print("✅ Imports successful! Ready to control your Duckiebot.")

## 2. Connecting to Your Duckiebot

You can connect to either a **simulated** robot (great for testing) or a **real** robot.

### For Simulation:
- Robot name: `"map_0/vehicle_0"` 
- Set `simulated=True`

### For Real Robot:
- Robot name: Your robot's hostname (e.g., `"db21j3"`)
- Set `simulated=False` (default)

In [ ]:
# Choose your robot type:
# For simulation:
ROBOT_NAME = "map_0/vehicle_0"
SIMULATED = True

# For real robot (uncomment and modify):
# ROBOT_NAME = "your_robot_name"  # Replace with your robot's name
# SIMULATED = False

# Create robot instance
robot = DB21J(ROBOT_NAME, simulated=SIMULATED)
print(f"🤖 Connected to {'simulated' if SIMULATED else 'real'} robot: {ROBOT_NAME}")

## 3. Making Your Robot Move! 🚗

Let's start with the most exciting part - making your robot move forward!

In [ ]:
def move_robot_forward(duration=2.0, speed=0.3):
    """
    Move the robot forward for a specified duration.
    
    Args:
        duration: How long to move (seconds)
        speed: Motor speed (0.0 to 1.0)
    """
    print(f"🚀 Moving forward at speed {speed} for {duration} seconds...")
    
    # Start the motors
    robot.motors.start()
    
    # Set both motors to the same speed (left, right)
    speeds: Tuple[float, float] = (speed, speed)
    
    start_time = time.time()
    while time.time() - start_time < duration:
        robot.motors.publish(speeds)
        time.sleep(0.1)  # Small delay between commands
    
    # Stop the robot
    robot.motors.publish((0.0, 0.0))
    robot.motors.stop()
    print("⛔ Stopped.")

# Try it out!
move_robot_forward(duration=2.0, speed=0.3)

## 4. Making Your Robot Turn 🔄

Now let's make the robot turn by setting different speeds for left and right motors:

In [ ]:
def turn_robot(direction="left", duration=1.0, speed=0.4):
    """
    Turn the robot left or right.
    
    Args:
        direction: "left" or "right"
        duration: How long to turn (seconds)
        speed: Motor speed (0.0 to 1.0)
    """
    print(f"🔄 Turning {direction} for {duration} seconds...")
    
    robot.motors.start()
    
    if direction == "left":
        # Left motor slower, right motor faster
        speeds = (speed * 0.2, speed)
    else:  # right
        # Left motor faster, right motor slower
        speeds = (speed, speed * 0.2)
    
    start_time = time.time()
    while time.time() - start_time < duration:
        robot.motors.publish(speeds)
        time.sleep(0.1)
    
    robot.motors.publish((0.0, 0.0))
    robot.motors.stop()
    print("⛔ Turn complete.")

# Try turning left and right
turn_robot("left", duration=1.0)
time.sleep(0.5)
turn_robot("right", duration=1.0)

## 5. Light Show! 💡✨

Let's make your robot's LEDs shine with a beautiful blinking pattern:

In [ ]:
def led_light_show(duration=8.0, frequency=1.4):
    """
    Create a blinking light pattern.
    
    Args:
        duration: How long to run the light show (seconds)
        frequency: Blinks per second
    """
    print(f"✨ Starting {duration}s light show at {frequency} Hz...")
    
    # Define colors
    off = RGBA(r=0, g=0, b=0, a=0.0)  # Lights off
    amber = RGBA(r=1, g=0.7, b=0, a=1.0)  # Amber/orange color
    blue = RGBA(r=0, g=0, b=1, a=1.0)  # Blue color
    
    # Create light patterns
    lights_amber = CarLights(
        front_left=amber, 
        front_right=amber, 
        back_right=amber, 
        back_left=amber
    )
    
    lights_blue = CarLights(
        front_left=blue, 
        front_right=blue, 
        back_right=blue, 
        back_left=blue
    )
    
    lights_off = CarLights(
        front_left=off, 
        front_right=off, 
        back_right=off, 
        back_left=off
    )
    
    # Start the light system
    robot.lights.start()
    
    # Create the pattern: amber -> blue -> off
    pattern = [lights_amber, lights_blue, lights_off]
    
    start_time = time.time()
    i = 0
    
    while time.time() - start_time < duration:
        current_lights = pattern[i % len(pattern)]
        robot.lights.publish(current_lights)
        time.sleep(1.0 / frequency)
        i += 1
    
    # Turn off lights and stop
    robot.lights.publish(lights_off)
    robot.lights.stop()
    print("💡 Light show complete!")

# Start the show!
led_light_show(duration=5.0, frequency=2.0)

## 6. Reading Sensor Data 📊

Let's read some sensor data from our robot:

In [ ]:
def read_wheel_encoders():
    """
    Read wheel encoder data to see how much the wheels have rotated.
    """
    print("📊 Reading wheel encoder data...")
    
    # Start the wheel encoder sensors
    robot.left_wheel_encoder.start()
    robot.right_wheel_encoder.start()
    
    # Wait a moment for data
    time.sleep(1.0)
    
    # Try to get the latest readings
    left_reading = robot.left_wheel_encoder.capture(block=True, timeout=2.0)
    right_reading = robot.right_wheel_encoder.capture(block=True, timeout=2.0)
    
    if left_reading is not None:
        print(f"🔧 Left wheel encoder: {left_reading}")
    else:
        print("⚠️ Could not read left wheel encoder")
        
    if right_reading is not None:
        print(f"🔧 Right wheel encoder: {right_reading}")
    else:
        print("⚠️ Could not read right wheel encoder")
    
    # Stop the sensors
    robot.left_wheel_encoder.stop()
    robot.right_wheel_encoder.stop()

# Try reading sensor data
read_wheel_encoders()

## 7. Getting Robot Pose 🧭

Let's find out where our robot is in the world:

In [ ]:
def get_robot_pose():
    """
    Get the robot's current position and orientation.
    """
    print("🧭 Getting robot pose...")
    
    robot.pose.start()
    
    # Capture the pose
    pose = robot.pose.capture(block=True, timeout=3.0)
    
    if pose is not None:
        position = pose["position"]
        rotation = pose["rotation"]
        
        print("📍 Robot Position:")
        print(f"   X: {position['x']:.3f}")
        print(f"   Y: {position['y']:.3f}")
        print(f"   Z: {position['z']:.3f}")
        
        print("🔄 Robot Orientation (quaternion):")
        print(f"   W: {rotation['w']:.3f}")
        print(f"   X: {rotation['x']:.3f}")
        print(f"   Y: {rotation['y']:.3f}")
        print(f"   Z: {rotation['z']:.3f}")
    else:
        print("⚠️ Could not get robot pose")
    
    robot.pose.stop()

# Get the current pose
get_robot_pose()

## 8. Reset Robot Position 🔄

For simulated robots, you can reset the position:

In [ ]:
def reset_robot_position():
    """
    Reset the robot to its initial position (simulation only).
    """
    if not SIMULATED:
        print("⚠️ Reset only works with simulated robots")
        return
        
    print("🔄 Resetting robot position...")
    
    robot.reset_flag.start()
    robot.reset_flag.publish(True)
    time.sleep(1.0)
    robot.reset_flag.stop()
    
    print("✅ Robot position reset!")
    
    # Show the new pose
    time.sleep(0.5)
    get_robot_pose()

# Reset if using simulation
if SIMULATED:
    reset_robot_position()

## 9. Put It All Together: A Simple Routine 🎭

Let's combine everything into a simple routine:

In [ ]:
def demo_routine():
    """
    A complete demo showing all robot capabilities.
    """
    print("🎭 Starting demo routine...")
    print("=" * 40)
    
    # 1. Get initial position
    print("\n1️⃣ Initial position:")
    get_robot_pose()
    
    # 2. Light show while moving
    print("\n2️⃣ Moving forward with lights:")
    # Start lights in background
    robot.lights.start()
    amber = RGBA(r=1, g=0.7, b=0, a=1.0)
    lights_on = CarLights(front_left=amber, front_right=amber, 
                         back_right=amber, back_left=amber)
    robot.lights.publish(lights_on)
    
    # Move forward
    move_robot_forward(duration=2.0, speed=0.3)
    
    # Turn off lights
    off = RGBA(r=0, g=0, b=0, a=0.0)
    lights_off = CarLights(front_left=off, front_right=off, 
                          back_right=off, back_left=off)
    robot.lights.publish(lights_off)
    robot.lights.stop()
    
    # 3. Turn around
    print("\n3️⃣ Turning around:")
    turn_robot("left", duration=2.0, speed=0.4)
    
    # 4. Final position
    print("\n4️⃣ Final position:")
    get_robot_pose()
    
    # 5. Victory light show
    print("\n5️⃣ Victory celebration:")
    led_light_show(duration=3.0, frequency=3.0)
    
    print("\n🎉 Demo complete! Your Duckiebot is ready for more adventures!")

# Run the demo!
demo_routine()

## 🎓 Congratulations!

You've successfully completed the Duckietown SDK Hello World tutorial! You now know how to:

✅ Connect to real and simulated Duckiebots  
✅ Control motors to move the robot  
✅ Create beautiful LED light patterns  
✅ Read sensor data  
✅ Get robot position and orientation  
✅ Reset robot state (simulation)  

## 🚀 Next Steps

Now that you've mastered the basics, here are some ideas for further exploration:

1. **Camera Vision**: Use `robot.camera` to capture images and process them
2. **Lane Following**: Implement algorithms to follow lane markings
3. **Obstacle Avoidance**: Use `robot.range_finder` for distance sensing
4. **Complex Behaviors**: Chain together movements to create autonomous behaviors
5. **Real Robot**: Try your code on a physical Duckiebot!

## 📚 Additional Resources

- [Duckietown Documentation](https://docs.duckietown.org/)
- [SDK Source Code](https://github.com/duckietown/duckietown-sdk)
- [Duckietown Community](https://www.duckietown.org/)

Happy coding with your Duckiebot! 🦆🤖✨